# KrishiMitra — Crop Disease Classifier (Model 1)

OCI Data Science notebook for the **Crop Disease Scanner** (Module 1).

- **Framework:** TensorFlow 2.x / Keras
- **Architecture:** MobileNetV3-Small (lightweight, fast inference, transfer learning)
- **Dataset:** PlantVillage — 38 classes, ~54K leaf images
- **Training:** OCI Data Science Job on a GPU shape (VM.GPU2.1)
- **Evaluation:** accuracy, precision/recall/F1 per class, confusion matrix
- **Deployment:** OCI Model Deployment (VM.Standard.E4.Flex, 2 OCPU) and/or an
  OCI Vision custom model trained from the same labelled set.

> Run on the `tensorflow28_p38_gpu_v1` conda env on a GPU shape. The dataset is
> expected in Object Storage and synced to `./PlantVillage` (see cell 2).

In [ ]:
import os
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import MobileNetV3Small

IMG_SIZE = (224, 224)
BATCH = 32
EPOCHS = 20
DATA_DIR = os.environ.get('PLANTVILLAGE_DIR', './PlantVillage')
print('TF', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))

## 1. Sync dataset from Object Storage
```bash
oci os object bulk-download -bn krishimitra-model-artifacts \
  --prefix datasets/PlantVillage/ --download-dir ./PlantVillage
```
Expected layout: `PlantVillage/<class_name>/*.jpg` (38 class folders).

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset='training', seed=42,
    image_size=IMG_SIZE, batch_size=BATCH)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset='validation', seed=42,
    image_size=IMG_SIZE, batch_size=BATCH)
class_names = train_ds.class_names
num_classes = len(class_names)
print(num_classes, 'classes')

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

## 2. Build model (transfer learning + augmentation)

In [ ]:
augment = models.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name='augment')

base = MobileNetV3Small(input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')
base.trainable = False  # freeze for the warm-up phase

inputs = layers.Input(shape=IMG_SIZE + (3,))
x = augment(inputs)
x = tf.keras.applications.mobilenet_v3.preprocess_input(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)
model = models.Model(inputs, outputs)
model.compile(optimizer=optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 3. Train (warm-up, then fine-tune)

In [ ]:
cbs = [
    callbacks.EarlyStopping(patience=4, restore_best_weights=True),
    callbacks.ModelCheckpoint('model-artifact/best.keras', save_best_only=True),
]
os.makedirs('model-artifact', exist_ok=True)
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=cbs)

# Fine-tune the top of the backbone
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False
model.compile(optimizer=optimizers.Adam(1e-5),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_ft = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=cbs)

## 4. Evaluation — accuracy + per-class precision/recall/F1

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

y_true, y_pred = [], []
for images, labels in val_ds:
    probs = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(probs, axis=1))
    y_true.extend(labels.numpy())

print(classification_report(y_true, y_pred, target_names=class_names))
cm = confusion_matrix(y_true, y_pred)
print('Confusion matrix shape:', cm.shape)
acc = (np.array(y_true) == np.array(y_pred)).mean()
print(f'Validation accuracy: {acc:.4f}  (target > 0.90)')

## 5. Export label map + save artifact

In [ ]:
import json
with open('model-artifact/class_names.json', 'w') as f:
    json.dump(class_names, f, indent=2)
model.save('model-artifact/disease_mobilenetv3.keras')
print('Saved Keras model + class_names.json')

# Register/deploy via ADS TensorFlowModel, OR upload the labelled set to
# OCI Vision Data Labeling and train a Vision custom model (used by the
# disease-classifier Function). The label_map below maps class folder names
# to the disease_lookup.json consumed by that Function.
label_map = {c: c.replace('___', ' ').replace('_', ' ') for c in class_names}
with open('model-artifact/label_map.json', 'w') as f:
    json.dump(label_map, f, ensure_ascii=False, indent=2)